# AI-Based Customer Support Query Classification and Response System

## AIML Academic Project — Complete Experimentation Notebook

This notebook demonstrates the complete machine learning pipeline for classifying customer support queries into predefined categories using traditional NLP and supervised learning techniques.

### Pipeline Overview:
1. Import Libraries
2. Load Dataset
3. Explore Dataset (EDA)
4. Data Cleaning
5. NLP Preprocessing
6. Class Distribution Visualization
7. Train/Test Split
8. Feature Extraction (BoW + TF-IDF)
9. Train ML Models
10. Evaluate Models
11. Compare Models
12. Select Best Model
13. Test Predictions

---
## 1. Import Libraries

We import the necessary Python libraries for:
- **Data handling**: pandas, numpy
- **NLP**: nltk (tokenization, stopwords, lemmatization)
- **Feature extraction**: scikit-learn's CountVectorizer and TfidfVectorizer
- **ML models**: LogisticRegression, MultinomialNB, LinearSVC, RandomForestClassifier
- **Evaluation**: accuracy, precision, recall, F1, confusion matrix
- **Visualization**: matplotlib, seaborn

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# NLP
import re
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Feature Extraction
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Train/Test Split
from sklearn.model_selection import train_test_split

# ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model saving
import joblib

# Settings
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

print('All libraries imported successfully!')

---
## 2. Load Dataset

We load the customer support dataset from a CSV file. The dataset contains two columns:
- **query**: The customer's text query
- **category**: The support category label

In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/customer_support.csv')

# Display first 10 rows
print('First 10 rows of the dataset:')
df.head(10)

---
## 3. Explore Dataset (Exploratory Data Analysis)

Before building any model, we need to understand our data:
- How many samples do we have?
- What columns exist?
- Are there missing values?
- Are there duplicate rows?
- How are the categories distributed?

In [ ]:
# Dataset shape
print(f'Dataset Shape: {df.shape}')
print(f'Number of samples: {df.shape[0]}')
print(f'Number of columns: {df.shape[1]}')

In [ ]:
# Column information
print('\nColumn Information:')
print(df.info())

In [ ]:
# Check for missing values
print('\nMissing Values:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f'Duplicate rows: {duplicate_count}')

# Remove duplicates if any
if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'After removing duplicates: {df.shape[0]} rows')

In [ ]:
# Number of unique categories
print(f'Number of categories: {df["category"].nunique()}')
print(f'\nCategories: {df["category"].unique().tolist()}')

In [ ]:
# Class distribution
print('\nSamples per category:')
print(df['category'].value_counts())

In [ ]:
# Basic text statistics
df['query_length'] = df['query'].apply(len)
df['word_count'] = df['query'].apply(lambda x: len(x.split()))

print('Text Statistics:')
print(f'  Average query length (chars): {df["query_length"].mean():.1f}')
print(f'  Average word count: {df["word_count"].mean():.1f}')
print(f'  Min words: {df["word_count"].min()}')
print(f'  Max words: {df["word_count"].max()}')

---
## 4. Class Distribution Visualization

Visualizing the class distribution helps us understand if the dataset is balanced or imbalanced. A balanced dataset means each category has roughly the same number of samples.

In [ ]:
# Plot class distribution
plt.figure(figsize=(10, 6))
category_counts = df['category'].value_counts()
colors = sns.color_palette('Set2', len(category_counts))

bars = plt.bar(category_counts.index, category_counts.values, color=colors)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.title('Dataset: Class Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')

# Add value labels
for bar, val in zip(bars, category_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 5. Data Cleaning & NLP Preprocessing

### Why is NLP preprocessing needed?

Raw text cannot be directly used by machine learning algorithms because:
- ML models work with **numbers**, not text
- Text contains noise like punctuation, special characters, varying cases
- Common words ("the", "is", "at") add no meaning but add dimensions
- Different word forms ("running", "runs", "ran") represent the same concept

### Our Preprocessing Pipeline:
1. **Lowercasing** — Normalize "PAYMENT" and "payment" to the same word
2. **Special Character Removal** — Remove punctuation like !, ?, #
3. **Whitespace Removal** — Clean extra spaces
4. **Tokenization** — Split text into individual words (tokens)
5. **Stopword Removal** — Remove common words that don't carry meaning
6. **Lemmatization** — Reduce words to their base/dictionary form

In [ ]:
# Initialize NLTK components
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Apply the full NLP preprocessing pipeline to a text string.
    
    Steps:
        1. Lowercase
        2. Remove special characters
        3. Remove extra whitespace
        4. Tokenize
        5. Remove stopwords
        6. Lemmatize
    """
    if not isinstance(text, str):
        return ''
    
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Step 3: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 4: Tokenize
    tokens = word_tokenize(text)
    
    # Step 5: Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    
    # Step 6: Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return ' '.join(tokens)

print('Preprocessing function defined successfully!')

In [ ]:
# Demonstrate preprocessing on sample queries
demo_queries = [
    'My PAYMENT has FAILED!!!',
    'Where is my order???',
    'I want my money back!!!',
    'The product I received is DAMAGED...',
    'I cannot login to my account',
]

print('Preprocessing Demonstration:')
print('=' * 60)
for q in demo_queries:
    cleaned = preprocess_text(q)
    print(f'  Original : {q}')
    print(f'  Processed: {cleaned}')
    print()

In [ ]:
# Apply preprocessing to the entire dataset
df['cleaned_query'] = df['query'].apply(preprocess_text)

# Show before and after
print('Before and After Preprocessing (first 5 rows):')
print('=' * 80)
for idx, row in df.head(5).iterrows():
    print(f'  Original : {row["query"]}')
    print(f'  Cleaned  : {row["cleaned_query"]}')
    print()

---
## 6. Train/Test Split

### Why do we split the data?

We split the dataset into **training set (80%)** and **testing set (20%)** because:
- The model learns patterns from the training data
- The testing data is used to evaluate how well the model generalizes to **unseen** data
- Without splitting, we would be testing on the same data we trained on (overfitting)

We use **stratification** to ensure all categories are represented proportionally in both sets.

In [ ]:
# Define features (X) and labels (y)
X = df['cleaned_query']
y = df['category']

# Split: 80% train, 20% test, with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples:  {len(X_test)}')
print(f'\nTraining class distribution:')
print(y_train.value_counts())
print(f'\nTesting class distribution:')
print(y_test.value_counts())

---
## 7. Feature Extraction

### Why can't we give text directly to ML models?

Machine learning algorithms work with **numerical data** (numbers), not text. We need to convert our text into numbers. Two common methods are:

### Bag of Words (BoW)
- Counts how many times each word appears in a document
- Creates a vector where each dimension = one unique word
- Simple and effective

### TF-IDF (Term Frequency - Inverse Document Frequency)
- Similar to BoW but assigns **weights** to words
- Words that are common across ALL documents get **lower** weight
- Words that are unique to specific documents get **higher** weight
- Better at capturing word importance

**IMPORTANT**: We fit the vectorizer ONLY on training data to prevent data leakage.

In [ ]:
# ===== Bag of Words (BoW) =====
print('=== Bag of Words Feature Extraction ===')

bow_vectorizer = CountVectorizer()

# Fit ONLY on training data, then transform
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print(f'Vocabulary size: {len(bow_vectorizer.vocabulary_)}')
print(f'Training matrix shape: {X_train_bow.shape}')
print(f'Testing matrix shape: {X_test_bow.shape}')
print(f'\nSample feature names: {bow_vectorizer.get_feature_names_out()[:15].tolist()}')

In [ ]:
# ===== TF-IDF =====
print('=== TF-IDF Feature Extraction ===')

tfidf_vectorizer = TfidfVectorizer()

# Fit ONLY on training data, then transform
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f'Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}')
print(f'Training matrix shape: {X_train_tfidf.shape}')
print(f'Testing matrix shape: {X_test_tfidf.shape}')
print(f'\nSample feature names: {tfidf_vectorizer.get_feature_names_out()[:15].tolist()}')

---
## 8. Train Machine Learning Models

We train **4 different ML algorithms** on **both BoW and TF-IDF** features:

| Model | How it works |
|---|---|
| **Logistic Regression** | Uses a linear function + sigmoid to calculate class probabilities |
| **Multinomial Naive Bayes** | Uses Bayes' theorem — assumes features are independent |
| **Support Vector Machine** | Finds the best hyperplane to separate classes |
| **Random Forest** | Ensemble of decision trees, each voting on the class |

This gives us **8 model-feature combinations** to compare.

In [ ]:
# Define the models
def get_models():
    return {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Naive Bayes': MultinomialNB(),
        'SVM': LinearSVC(max_iter=2000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    }

# Store all results
all_results = []
trained_models = {}

In [ ]:
# ===== EXPERIMENT A: Train on BoW Features =====
print('=' * 60)
print('EXPERIMENT A: Training on Bag of Words Features')
print('=' * 60)

models_bow = get_models()

for name, model in models_bow.items():
    print(f'\nTraining {name} on BoW...')
    model.fit(X_train_bow, y_train)
    y_pred = model.predict(X_test_bow)
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    all_results.append({
        'Feature Method': 'BoW',
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1 Score': round(f1, 4),
    })
    
    trained_models[f'BoW_{name}'] = {'model': model, 'y_pred': y_pred}
    
    print(f'  Accuracy: {acc:.4f} | F1: {f1:.4f}')

In [ ]:
# ===== EXPERIMENT B: Train on TF-IDF Features =====
print('=' * 60)
print('EXPERIMENT B: Training on TF-IDF Features')
print('=' * 60)

models_tfidf = get_models()

for name, model in models_tfidf.items():
    print(f'\nTraining {name} on TF-IDF...')
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    all_results.append({
        'Feature Method': 'TF-IDF',
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1 Score': round(f1, 4),
    })
    
    trained_models[f'TFIDF_{name}'] = {'model': model, 'y_pred': y_pred}
    
    print(f'  Accuracy: {acc:.4f} | F1: {f1:.4f}')

---
## 9. Model Evaluation

### Evaluation Metrics Explained:

- **Accuracy**: What % of all predictions were correct?
- **Precision**: Of all items predicted as class X, what % actually belong to class X?
- **Recall**: Of all actual class X items, what % did the model correctly identify?
- **F1 Score**: Harmonic mean of Precision and Recall — balances both metrics
- **Confusion Matrix**: Shows exactly which classes are being confused with each other

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame(all_results)
comparison_df = comparison_df.sort_values('F1 Score', ascending=False).reset_index(drop=True)

print('MODEL COMPARISON TABLE')
print('=' * 80)
comparison_df

In [ ]:
# Print detailed classification reports for the top models
print('\n' + '=' * 60)
print('DETAILED CLASSIFICATION REPORTS')
print('=' * 60)

for key, data in trained_models.items():
    parts = key.split('_', 1)
    feature = 'BoW' if parts[0] == 'BoW' else 'TF-IDF'
    model_name = parts[1]
    print(f'\n--- {model_name} ({feature}) ---')
    print(classification_report(y_test, data['y_pred'], zero_division=0))

In [ ]:
# Plot confusion matrices for all models
labels = sorted(y_test.unique())

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
fig.suptitle('Confusion Matrices: All Model-Feature Combinations', fontsize=16, fontweight='bold')

for idx, (key, data) in enumerate(trained_models.items()):
    row = idx // 4
    col = idx % 4
    ax = axes[row][col]
    
    cm = confusion_matrix(y_test, data['y_pred'], labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax)
    
    parts = key.split('_', 1)
    feature = 'BoW' if parts[0] == 'BoW' else 'TF-IDF'
    ax.set_title(f'{parts[1]} ({feature})', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 10. Model Comparison Visualization

Visual comparison makes it easy to see which model-feature combination performs best.

In [ ]:
# Accuracy comparison bar chart
comparison_df_plot = comparison_df.copy()
comparison_df_plot['Label'] = comparison_df_plot['Model'] + ' (' + comparison_df_plot['Feature Method'] + ')'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy
colors = sns.color_palette('viridis', len(comparison_df_plot))
axes[0].barh(comparison_df_plot['Label'], comparison_df_plot['Accuracy'], color=colors)
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Model Comparison: Accuracy', fontweight='bold')
axes[0].set_xlim(0, 1.0)
for i, val in enumerate(comparison_df_plot['Accuracy']):
    axes[0].text(val + 0.01, i, f'{val:.4f}', va='center')

# F1 Score
axes[1].barh(comparison_df_plot['Label'], comparison_df_plot['F1 Score'], color=colors)
axes[1].set_xlabel('F1 Score')
axes[1].set_title('Model Comparison: F1 Score', fontweight='bold')
axes[1].set_xlim(0, 1.0)
for i, val in enumerate(comparison_df_plot['F1 Score']):
    axes[1].text(val + 0.01, i, f'{val:.4f}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# BoW vs TF-IDF side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, method in enumerate(['BoW', 'TF-IDF']):
    subset = comparison_df_plot[comparison_df_plot['Feature Method'] == method]
    x = np.arange(len(subset))
    width = 0.35
    
    axes[idx].bar(x - width/2, subset['Accuracy'].values, width, label='Accuracy', color='steelblue')
    axes[idx].bar(x + width/2, subset['F1 Score'].values, width, label='F1 Score', color='coral')
    axes[idx].set_title(f'{method} Feature Extraction', fontweight='bold')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(subset['Model'].values, rotation=30, ha='right')
    axes[idx].legend()
    axes[idx].set_ylim(0, 1.1)

plt.suptitle('BoW vs TF-IDF: Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Best Model Selection

We select the best model based on the **F1 Score**, which balances both precision and recall.

In [ ]:
# Find the best model
best_row = comparison_df.loc[comparison_df['F1 Score'].idxmax()]

print('=' * 50)
print('BEST MODEL SELECTED')
print('=' * 50)
print(f'  Model:              {best_row["Model"]}')
print(f'  Feature Method:     {best_row["Feature Method"]}')
print(f'  Accuracy:           {best_row["Accuracy"]}')
print(f'  Precision:          {best_row["Precision"]}')
print(f'  Recall:             {best_row["Recall"]}')
print(f'  F1 Score:           {best_row["F1 Score"]}')

---
## 12. Test Predictions on Sample Queries

Let's test the best model on new, unseen queries to verify it works correctly.

In [ ]:
# Get the best model and vectorizer
best_feature = best_row['Feature Method']
best_model_name = best_row['Model']

if best_feature == 'BoW':
    best_key = f'BoW_{best_model_name}'
    best_vectorizer = bow_vectorizer
else:
    best_key = f'TFIDF_{best_model_name}'
    best_vectorizer = tfidf_vectorizer

best_model = trained_models[best_key]['model']

# Predefined responses
RESPONSES = {
    'Delivery': 'Your query has been identified as a delivery-related issue. Please check your order tracking details.',
    'Payment': 'Your query appears to be related to payment. Please verify the transaction status and payment details.',
    'Refund': 'Your query appears to be related to a refund. Please check your refund status using the order details.',
    'Cancellation': 'Your query appears to be related to order cancellation. Please check eligibility for cancellation.',
    'Product Issue': 'Your query appears to concern a product issue. Please provide order details and describe the issue.',
    'Complaint': 'We have identified your query as a complaint. Please provide relevant details for investigation.',
    'Account Issue': 'Your query appears to be related to your account. Please verify your login information.',
}

# Test queries
test_queries = [
    'My order has not arrived yet',
    'My payment failed',
    'I want my money back',
    'I want to cancel my order',
    'The product I received is damaged',
    'I am unhappy with your service',
    'I cannot login to my account',
    'Where is my package it was supposed to come yesterday',
    'You charged me twice for the same item',
    'The item looks completely different from what was shown',
]

print('PREDICTION RESULTS')
print('=' * 70)
for query in test_queries:
    # Preprocess
    cleaned = preprocess_text(query)
    # Vectorize
    vector = best_vectorizer.transform([cleaned])
    # Predict
    category = best_model.predict(vector)[0]
    # Get response
    response = RESPONSES.get(category, 'Thank you for your query.')
    
    print(f'\n  Query:    "{query}"')
    print(f'  Category: {category}')
    print(f'  Response: {response}')

---
## 13. Save Results

Save the comparison table to CSV for documentation.

In [ ]:
# Save comparison table
comparison_df.to_csv('../results/model_comparison.csv', index=False)
print('Model comparison saved to: results/model_comparison.csv')
print()
print(comparison_df.to_string(index=False))

---
## Summary

In this notebook, we demonstrated the complete AIML pipeline:

1. **Loaded** a customer support dataset with 7 categories
2. **Explored** the data — checked shape, missing values, duplicates, distribution
3. **Preprocessed** text using NLP — lowercasing, tokenization, stopword removal, lemmatization
4. **Split** data into 80% train / 20% test with stratification
5. **Extracted** features using both **Bag of Words** and **TF-IDF**
6. **Trained** 4 ML models × 2 feature methods = **8 combinations**
7. **Evaluated** all models using accuracy, precision, recall, F1 score
8. **Visualized** results with confusion matrices and comparison charts
9. **Selected** the best model based on actual F1 scores
10. **Tested** predictions on new sample queries

### Key Concepts Demonstrated:
- Traditional NLP text preprocessing
- Feature engineering for text (BoW vs TF-IDF)
- Supervised multiclass classification
- Model evaluation and comparison
- Preventing data leakage (fit vectorizer on train only)